### 1.Import Libraries & Load Dataset

In [17]:
import pandas as pd
import numpy as np
import re
import string
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download('stopwords')


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\RANI\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

### 2. Load Your Dataset

In [18]:
df = pd.read_csv("twitter.csv")
df = df[["tweet", "class"]]   # keep only useful columns


### 3.class dataset → Binary Sentiment

class 0 = hate speech → negative (0)
class 1 = normal → positive (1)

In [19]:
# 2 = normal (positive), 0/1 = hate/offensive (negative)
df["sentiment"] = df["class"].apply(lambda x: 1 if x == 2 else 0)


### 4.Balance the Dataset

In [20]:
pos_df = df[df["sentiment"] == 1]
neg_df = df[df["sentiment"] == 0]

min_size = min(len(pos_df), len(neg_df))

pos_df_bal = pos_df.sample(min_size, random_state=1)
neg_df_bal = neg_df.sample(min_size, random_state=1)

df_balanced = pd.concat([pos_df_bal, neg_df_bal]).sample(frac=1, random_state=1)

train_x = list(df_balanced["tweet"])
train_y = list(df_balanced["sentiment"])


### 5. Preprocess Tweets

In [21]:
stopwords_english = stopwords.words("english")
stemmer = PorterStemmer()

def process_tweet(tweet):
    tweet = tweet.lower()
    tweet = re.sub(r'https?://\S+', '', tweet)  # remove URLs
    tweet = re.sub(r'@\w+', '', tweet)          # remove mentions
    tweet = re.sub(r'#[A-Za-z0-9_]+', '', tweet)  # remove hashtags
    
    words = re.findall(r'\w+', tweet)

    clean_words = []
    for w in words:
        if w not in stopwords_english and w not in string.punctuation:
            clean_words.append(stemmer.stem(w))
    return clean_words


### 6. Build Frequency Dictionary

In [22]:
def build_freqs(tweets, labels):
    freqs = {}
    for tweet, y in zip(tweets, labels):
        words = process_tweet(tweet)
        for word in words:
            pair = (word, y)
            freqs[pair] = freqs.get(pair, 0) + 1
    return freqs

freqs = build_freqs(train_x, train_y)


### 7. Train Naive Bayes Model (INCLUDES logprior & loglikelihood)

In [23]:
def train_naive_bayes(freqs, train_x, train_y):
    loglikelihood = {}
    pos_total = 0
    neg_total = 0

    vocab = set([word for (word, label) in freqs.keys()])

    # Count pos & neg words
    for (word, label), count in freqs.items():
        if label == 1:
            pos_total += count
        else:
            neg_total += count

    # Compute loglikelihood for each word
    for word in vocab:
        freq_pos = freqs.get((word, 1), 0)
        freq_neg = freqs.get((word, 0), 0)

        p_w_pos = (freq_pos + 1) / (pos_total + len(vocab))
        p_w_neg = (freq_neg + 1) / (neg_total + len(vocab))

        loglikelihood[word] = np.log(p_w_pos / p_w_neg)

    # Compute logprior = log(P(positive)/P(negative))
    pos_count = sum(train_y)
    neg_count = len(train_y) - pos_count
    logprior = np.log(pos_count / neg_count)

    return logprior, loglikelihood


### 8.Generate logprior & loglikelihood

In [24]:
logprior, loglikelihood = train_naive_bayes(freqs, train_x, train_y)

print("logprior:", logprior)
print("Sample loglikelihood:", list(loglikelihood.items())[:5])


logprior: 0.0
Sample loglikelihood: [('225', np.float64(0.6231608010517742)), ('ht', np.float64(0.6231608010517742)), ('christin', np.float64(-1.1685986681762808)), ('cri', np.float64(-0.5554941952898721)), ('6month', np.float64(-0.7631335600681164))]


In [25]:
logprior, loglikelihood = train_naive_bayes(freqs, train_x, train_y)


### 9. Prediction Function

In [26]:
def predict_naive_bayes(tweet, logprior, loglikelihood):
    words = process_tweet(tweet)
    score = logprior
    
    for word in words:
        if word in loglikelihood:
            score += loglikelihood[word]

    return 1 if score > 0 else 0


### 10.Accuracy Test

In [27]:
correct = 0

for tweet, label in zip(train_x, train_y):
    if predict_naive_bayes(tweet, logprior, loglikelihood) == label:
        correct += 1

accuracy = correct / len(train_y)
print("Model Accuracy:", accuracy)


Model Accuracy: 0.961085755464809


### 11.Predict Your Own Tweet

In [32]:
my_tweet = "I love this! Amazing experience!"
print("Prediction:", predict_naive_bayes(my_tweet, logprior, loglikelihood))
my_tweet = "I hate this! Worst experience ever!"
print("Prediction:", predict_naive_bayes(my_tweet, logprior, loglikelihood))

Prediction: 1
Prediction: 1


In [33]:

print("Prediction:", predict_naive_bayes("You are stupid", logprior, loglikelihood))


Prediction: 0


In [34]:
my_tweet = "I love this! Amazing experience!"
print("Prediction:", predict_naive_bayes(my_tweet, logprior, loglikelihood))

Prediction: 1
